# PestCLEF 2026 – Knowledge-graph extraction pipeline
**Pipeline:** CRF + lexicon NER → mention clustering into concepts → schema-constrained candidate pairs → LightGBM scoring → per-document expected-F1 edge selection → `submission.csv`.

Metric: per-document edge F1 (empty gold graph scores 1 only for an empty prediction).
All validation numbers are **5-fold out-of-fold** over train+dev (165 docs). Runtime ≈ 5–8 min on 1 CPU.

Set the paths in the config cell below, then *Run All*.

In [ ]:
#!pip install -q lightgbm sklearn-crfsuite pycountry

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

## 0. Config

In [2]:
import os
U     = 'D:/uni/DDfCIS'
D     = 'D:/uni/DDfCIS/EPOP_documents'
OUT   = 'D:/uni/DDfCIS'
CACHE = './cache'

# normalise: forward slashes + exactly one trailing slash
fix = lambda p: p.replace('\\', '/').rstrip('/') + '/'
U, D, OUT, CACHE = map(fix, (U, D, OUT, CACHE))
os.makedirs(OUT, exist_ok=True); os.makedirs(CACHE, exist_ok=True)

# fail early with a useful message
for p in (U + 'PestCLEF-2026_train.json', D + 'train', D + 'dev', D + 'test'):
    print(('OK      ' if os.path.exists(p) else 'MISSING ') + p)
assert os.path.isdir(D + 'train'), 'D is wrong - see MISSING lines above'

OK      D:/uni/DDfCIS/PestCLEF-2026_train.json
OK      D:/uni/DDfCIS/EPOP_documents/train
OK      D:/uni/DDfCIS/EPOP_documents/dev
OK      D:/uni/DDfCIS/EPOP_documents/test


## 1. Data loading & official metric

In [3]:
import json,re,collections,pickle,math,time
import numpy as np
def load(split):
    docs=json.load(open(f'{U}PestCLEF-2026_{split}.json'))
    for x in docs:
        x['text']=open(f'{D}{split}/{x["doc_id"]}.txt',encoding='utf-8').read()
    return docs
def norm(s): return re.sub(r'\s+',' ',s.strip().lower())

def gold_edges(doc):
    """list of (frozenset subj forms, pred, frozenset obj forms)"""
    return [(frozenset(norm(f) for f in e['subject']),e['predicate'],frozenset(norm(f) for f in e['object'])) for e in doc.get('knowledge_graph',[])]

def f1_doc(pred, gold):
    """pred: list of (s,p,o) strings ; gold: list from gold_edges"""
    if not gold: return 1.0 if not pred else 0.0
    matched=set(); fp=0
    for s,p,o in set((norm(s),p,norm(o)) for s,p,o in pred):
        hit=[i for i,(gs,gp,go) in enumerate(gold) if gp==p and s in gs and o in go]
        if hit: matched.update(hit[:1]) if False else None; 
        if hit:
            matched.add(hit[0]) if hit[0] not in matched else matched.add(hit[0])
        else: fp+=1
    # dedup: preds matching same ref edge count once
    tp_edges=set()
    for s,p,o in set((norm(s),p,norm(o)) for s,p,o in pred):
        for i,(gs,gp,go) in enumerate(gold):
            if gp==p and s in gs and o in go: tp_edges.add(i); break
    tp=len(tp_edges); fn=len(gold)-tp
    if tp==0: return 0.0
    P=tp/(tp+fp); R=tp/(tp+fn); return 2*P*R/(P+R)
def score(preds, docs):
    """preds: {doc_id: [(s,p,o)]}"""
    xs=[f1_doc(preds.get(d['doc_id'],[]),gold_edges(d)) for d in docs]
    return sum(xs)/len(xs)


In [4]:
tr, dv, te = load('train'), load('dev'), load('test'); allx = tr + dv
print(len(tr), len(dv), len(te))

110 55 82


## 2. Stage 1 – Named-entity recognition (CRF with jack-knifed gazetteer features)

In [5]:
import sklearn_crfsuite, pycountry
TOK=re.compile(r"[A-Za-z\u00C0-\u024F]+(?:[-'][A-Za-z\u00C0-\u024F]+)*|\d+|[^\w\s]",re.U)
def tokenize(text):
    return [(m.group(),m.start(),m.end()) for m in TOK.finditer(text)]
MONTHS='january february march april may june july august september october november december spring summer autumn winter fall'.split()
STATIC_LOC=set()
for c in pycountry.countries:
    STATIC_LOC.add(c.name.lower()); 
    if hasattr(c,'common_name'): STATIC_LOC.add(c.common_name.lower())
for s in pycountry.subdivisions: STATIC_LOC.add(s.name.lower())
STATIC_LOC={x for x in STATIC_LOC if len(x)>3}
STATIC_LOC|={'europe','asia','africa','america','north america','south america','mediterranean','eu','european union','united states','usa','uk','china'}

def build_lexicon(docs,min_rate=0.0):
    lex=collections.defaultdict(collections.Counter)
    for d in docs:
        for e in d['text_bound_annotations']['entities']:
            toks=[t[0].lower() for t in tokenize(e['form'])]
            if toks: lex[tuple(toks)][e['type']]+=1
    return {k:v.most_common(1)[0][0] for k,v in lex.items()}
def gaz_tags(toks,lex):
    """longest match tagging -> list of (type,'B'/'I') or None"""
    low=[t[0].lower() for t in toks]; out=[None]*len(toks)
    maxlen=max((len(k) for k in lex),default=1); i=0
    while i<len(low):
        done=False
        for L in range(min(maxlen,len(low)-i),0,-1):
            k=tuple(low[i:i+L])
            if k in lex:
                for j in range(L): out[i+j]=(lex[k],'B' if j==0 else 'I')
                i+=L; done=True; break
        if not done: i+=1
    return out
def shape(w):
    s=re.sub('[A-Z]','X',w); s=re.sub('[a-z]','x',s); s=re.sub(r'\d','d',s)
    return re.sub(r'(.)\1{2,}',r'\1\1',s)
def feats(toks,gaz,static):
    F=[]
    for i,(w,a,b) in enumerate(toks):
        lw=w.lower()
        f={'b':1,'lw':lw,'sh':shape(w),'p2':lw[:2],'p3':lw[:3],'s2':lw[-2:],'s3':lw[-3:],'s4':lw[-4:],'ti':w.istitle(),'up':w.isupper(),'dg':w.isdigit(),
           'mon':lw in MONTHS,'sloc':static[i]}
        g=gaz[i]; f['gz']=(g[0]+'-'+g[1]) if g else 'O'
        for o in (-2,-1,1,2):
            j=i+o
            if 0<=j<len(toks):
                w2=toks[j][0]; l2=w2.lower(); f[f'{o}lw']=l2; f[f'{o}sh']=shape(w2); f[f'{o}ti']=w2.istitle()
                g2=gaz[j]; f[f'{o}gz']=(g2[0]+'-'+g2[1]) if g2 else 'O'
            else: f[f'{o}lw']='<pad>'
        F.append(f)
    return F
def static_loc(toks):
    low=[t[0].lower() for t in toks]; out=[False]*len(toks); i=0
    while i<len(low):
        hit=False
        for L in (4,3,2,1):
            if ' '.join(low[i:i+L]) in STATIC_LOC and i+L<=len(low):
                for j in range(L): out[i+j]=True
                i+=L; hit=True; break
        if not hit: i+=1
    return out
def bio(doc,toks):
    tags=['O']*len(toks)
    for e in doc['text_bound_annotations']['entities']:
        for si,(a,b) in enumerate(e['offsets']):
            idx=[i for i,t in enumerate(toks) if t[1]>=a and t[2]<=b]
            if not idx: 
                idx=[i for i,t in enumerate(toks) if t[2]>a and t[1]<b]
            for k,i in enumerate(idx):
                tags[i]=('B-' if k==0 else 'I-')+e['type']
    return tags
def decode(toks,tags,text):
    ms=[];cur=None
    for i,t in enumerate(tags):
        if t=='O':
            if cur: ms.append(cur);cur=None
        elif t.startswith('B-') or cur is None or cur[0]!=t[2:]:
            if cur: ms.append(cur)
            cur=[t[2:],i,i]
        else: cur[2]=i
    if cur: ms.append(cur)
    return [{'type':ty,'start':toks[i][1],'end':toks[j][2],'form':text[toks[i][1]:toks[j][2]]} for ty,i,j in ms]
def sent_feats(doc,lex):
    toks=tokenize(doc['text']); return toks,feats(toks,gaz_tags(toks,lex),static_loc(toks))
def jackknife_data(docs,k=5):
    X,Y=[],[]
    folds=[docs[i::k] for i in range(k)]
    for fi in range(k):
        lex=build_lexicon([d for j,f in enumerate(folds) if j!=fi for d in f])
        for d in folds[fi]:
            toks,F=sent_feats(d,lex); X.append(chunk(F)); Y.append(chunk(bio(d,toks)))
    return X,Y
def chunk(seq,n=400):
    return seq  # whole doc as a sequence
def train_crf(docs,c1=0.05,c2=0.05,iters=120):
    # sequences: split into windows of 300 tokens for speed
    X,Y=[],[]
    folds=[docs[i::5] for i in range(5)]
    for fi in range(5):
        lex=build_lexicon([d for j,f in enumerate(folds) if j!=fi for d in f])
        for d in folds[fi]:
            toks,F=sent_feats(d,lex); T=bio(d,toks)
            for s in range(0,len(F),300): X.append(F[s:s+300]);Y.append(T[s:s+300])
    crf=sklearn_crfsuite.CRF(algorithm='lbfgs',c1=c1,c2=c2,max_iterations=iters,all_possible_transitions=True)
    crf.fit(X,Y)
    return crf,build_lexicon(docs)
def predict_mentions(crf,lex,doc):
    toks,F=sent_feats(doc,lex); tags=[]
    for s in range(0,len(F),300): tags+=crf.predict_single(F[s:s+300])
    return decode(toks,tags,doc['text'])
def ent_eval(pred,docs_gold,doc):
    G={(e['offsets'][0][0],e['offsets'][-1][1],e['type']) for e in doc['text_bound_annotations']['entities']}
    P={(m['start'],m['end'],m['type']) for m in pred}
    return G,P


### Lexicon union
High-precision dictionary matches added on top of CRF mentions (non-overlapping).

In [6]:
def form_stats(docs):
    ann=collections.defaultdict(collections.Counter)
    for d in docs:
        for e in d['text_bound_annotations']['entities']: ann[norm(e['form'])][e['type']]+=1
    occ=collections.Counter()
    big=' \n '.join(d['text'].lower() for d in docs)
    stats={}
    for f,c in ann.items():
        if len(f)<3: continue
        n=len(re.findall(r'(?<![\w])'+re.escape(f)+r'(?![\w])',big))
        t,k=c.most_common(1)[0]
        stats[f]=(t,sum(c.values())/max(1,n),sum(c.values()),k/sum(c.values()))
    return stats
def lex_mentions(doc,stats,min_rate=0.5):
    out=[];low=doc['text'].lower()
    for f,(t,rate,cnt,tp) in stats.items():
        if rate<min_rate or tp<0.7: continue
        for m in re.finditer(r'(?<![\w])'+re.escape(f)+r'(?![\w])',low):
            out.append({'type':t,'start':m.start(),'end':m.end(),'form':doc['text'][m.start():m.end()]})
    return out
def union(crf_ms,lex_ms):
    out=list(crf_ms)
    occupied=[(m['start'],m['end']) for m in crf_ms]
    # add lexicon mention if it doesn't overlap a CRF mention; prefer longer lex ones first
    for m in sorted(lex_ms,key=lambda m:-(m['end']-m['start'])):
        if all(m['end']<=a or m['start']>=b for a,b in occupied):
            out.append(m);occupied.append((m['start'],m['end']))
    return sorted(out,key=lambda m:m['start'])


## 3. Stage 2 – Concept clustering, candidate generation, features

In [7]:
SIGS=[('Located_in','Pest','Location'),('Located_in','Plant','Location'),('Located_in','Disease','Location'),('Located_in','Vector','Location'),('Located_in','Dissemination_pathway','Location'),
('Found_on','Pest','Plant'),('Found_on','Vector','Plant'),('Found_on','Pest','Dissemination_pathway'),('Found_on','Vector','Dissemination_pathway'),
('Affects','Disease','Plant'),('Occurs_on','Pest','Date'),('Occurs_on','Plant','Date'),('Occurs_on','Disease','Date'),('Occurs_on','Vector','Date'),('Occurs_on','Dissemination_pathway','Date'),
('Causes','Pest','Disease'),('Transmits','Vector','Pest'),('Transmits','Vector','Disease'),('Dispersed_by','Pest','Dissemination_pathway'),('Dispersed_by','Disease','Dissemination_pathway')]
PREDS=sorted({s[0] for s in SIGS})
def stem(f):
    f=norm(f); f=re.sub(r'^(the|a|an)\s+','',f); return f
def build_ref_map(docs):
    """form -> normalization ref (most common) from gold annotations"""
    m=collections.defaultdict(collections.Counter)
    for d in docs:
        for e in d['text_bound_annotations']['entities']:
            for n in e['normalizations']: m[(stem(e['form']),e['type'])][n['reference']]+=1
    return {k:v.most_common(1)[0][0] for k,v in m.items()}
def cluster(mentions,text,refmap):
    """returns list of concepts: dict(type, forms Counter, mentions[list])"""
    # acronym detection: "Full Name (ACR)"
    acr={}
    for m in re.finditer(r'([A-Z][\w\-]*(?:\s+[\w\-]+){0,6}?)\s*\(\s*([A-Z][A-Za-z0-9\-]{1,10})\s*[;,)]',text): acr[m.group(2).lower()]=stem(m.group(1))
    groups={}
    for m in mentions:
        f=stem(m['form']); key=(m['type'],refmap.get((f,m['type'])) or acr.get(f) or f)
        # fold acronyms to their expanded form's ref
        if key[1]==f and f in acr:
            key=(m['type'],refmap.get((acr[f],m['type'])) or acr[f])
        groups.setdefault(key,[]).append(m)
    # merge groups whose key is the form of another group's ref via form containment of identical text
    C=[]
    for (ty,k),ms in groups.items():
        forms=collections.Counter(m['form'] for m in ms)
        C.append({'type':ty,'key':k,'forms':forms,'ms':ms})
    return C
def sent_bounds(text):
    b=[0]
    for m in re.finditer(r'(?<=[.!?])\s+(?=[A-Z0-9(])|\n+',text): b.append(m.end())
    return np.array(b)
def para_bounds(text):
    return np.array([0]+[m.end() for m in re.finditer(r'\n+',text)])
STOP=set('the a an'.split())
def words(s): return [w for w in re.findall(r'[a-z]+',s.lower()) if w not in STOP]
class Priors:
    """document-level counts used for memorisation features; supports leave-one-doc-out"""
    def __init__(s,docs):
        s.docs={}; s.tot=collections.defaultdict(collections.Counter)
        for d in docs:
            c=collections.defaultdict(collections.Counter)
            for sub,p,obj in gold_edges(d):
                for a in sub:
                    c['A'][(a,p)]+=1
                    for b in obj: c['AB'][(a,b,p)]+=1
                for b in obj: c['B'][(b,p)]+=1
            tl=d['text'].lower(); c['seen']=collections.Counter()
            s.docs[d['doc_id']]=c
            for k,v in c.items():
                for kk,vv in v.items(): s.tot[k][kk]+=vv
    def get(s,kind,key,exclude=None):
        v=s.tot[kind].get(key,0)
        if exclude is not None and exclude in s.docs: v-=s.docs[exclude][kind].get(key,0)
        return v
def candidates(doc,mentions,refmap,priors,exclude,labels=True,trigger_vocab=None):
    text=doc['text']; C=cluster(mentions,text,refmap)
    sb=sent_bounds(text); pb=para_bounds(text); n=len(text) or 1
    title_end=text.find('\n') if '\n' in text[:400] else 0
    bytype=collections.defaultdict(list)
    for ci,c in enumerate(C):
        c['canon']=c['forms'].most_common(1)[0][0]
        c['nm']=len(c['ms']); c['first']=min(m['start'] for m in c['ms'])
        c['sent']=[int(np.searchsorted(sb,m['start'],side='right')) for m in c['ms']]
        c['para']=[int(np.searchsorted(pb,m['start'],side='right')) for m in c['ms']]
        bytype[c['type']].append(ci)
    for ty,l in bytype.items():
        l.sort(key=lambda i:-C[i]['nm'])
        for r,i in enumerate(l): C[i]['rank']=r
    gold=gold_edges(doc) if labels else []
    rows=[];meta=[]
    for si,(p,ta,tb) in enumerate(SIGS):
        for i in bytype.get(ta,[]):
            for j in bytype.get(tb,[]):
                A,B=C[i],C[j]
                best=None
                for ma in A['ms']:
                    for mb in B['ms']:
                        dist=abs(ma['start']-mb['start'])
                        if best is None or dist<best[0]: best=(dist,ma,mb)
                dist,ma,mb=best
                sd=min(abs(x-y) for x in A['sent'] for y in B['sent'])
                pdist=min(abs(x-y) for x in A['para'] for y in B['para'])
                lo,hi=sorted((ma['end'],mb['end'])); a2,b2=(ma,mb) if ma['start']<mb['start'] else (mb,ma)
                between=text[a2['end']:b2['start']] if a2['end']<=b2['start'] else ''
                bw=words(between[:300]); ctx=words(text[max(0,a2['start']-60):a2['start']]+' '+text[b2['end']:b2['end']+60])
                ka,kb=stem(A['canon']),stem(B['canon'])
                f=[si,PREDS.index(p),A['nm'],B['nm'],math.log1p(dist),sd,pdist,int(sd==0),int(pdist==0),
                   A['first']/n,B['first']/n,int(A['first']<=title_end),int(B['first']<=title_end),A['rank'],B['rank'],
                   len(bytype[ta]),len(bytype[tb]),n/1000,len(A['ms'])*len(B['ms']),int(ma['start']<mb['start']),len(bw),
                   sum(1 for x in A['ms'] if True)/max(1,len(mentions)),
                   priors.get('AB',(ka,kb,p),exclude),priors.get('A',(ka,p),exclude),priors.get('B',(kb,p),exclude),
                   int(',' in between),int('(' in between or ')' in between),int(';' in between),
                   len(A['forms']),len(B['forms'])]
                tv=trigger_vocab or []
                bs=set(bw); cs=set(ctx)
                f+= [int(w in bs) for w in tv]+[int(w in cs) for w in tv]
                y=0
                if labels:
                    sa=norm(A['canon']); sb_=norm(B['canon'])
                    y=int(any(g[1]==p and sa in g[0] and sb_ in g[2] for g in gold))
                rows.append(f); meta.append((doc['doc_id'],p,A['canon'],B['canon'],y,ta,tb))
    if not rows: return np.zeros((0,0),dtype=np.float32),meta
    return np.array(rows,dtype=np.float32),meta


## 4. Stage 3 – LightGBM scorer & per-document edge selection

In [8]:
import lightgbm as lgb
def select(items,rho=0.7,empty_t=0.0,floor=0.0,scale=1.0):
    """items: list of (edge,p). Choose top-k maximizing approx expected F1 = 2*sum(p_topk)/(k+G), G~sum(p_all)/rho"""
    if not items: return []
    items=sorted(items,key=lambda x:-x[1]); ps=np.array([min(1,x[1]*scale) for x in items])
    G=max(1.0,ps.sum()/rho); best=(0,0); cs=np.cumsum(ps)
    for k in range(1,len(ps)+1):
        v=2*cs[k-1]/(k+G)
        if v>best[0]: best=(v,k)
    if best[1]==0 or ps[0]<empty_t: return []
    return [e for e,p in items[:best[1]] if p>=floor]


In [9]:
TV = sorted(set('in on by from since during and of found detected reported report infect infected infection affect affects affected cause causes caused agent vector vectors transmit transmitted transmission spread spreads outbreak first confirmed present introduced imported seed seeds plant plants fruit fruits trade host hosts crop crops symptoms damage pathogen insect disease virus bacterium also not other new eradicated no was is were'.split()))
PAR = dict(objective='binary', learning_rate=0.05, num_leaves=15, min_data_in_leaf=20, feature_fraction=0.7,
           bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0, verbose=-1, num_threads=1)
def fit(X, y, rounds=250, seed=0):
    P = dict(PAR, seed=seed, bagging_seed=seed, feature_fraction_seed=seed)
    return lgb.train(P, lgb.Dataset(X, y, categorical_feature=[0, 1]), rounds)
def build(docs, M, priors, refmap, labels=True, excl=True):
    X, meta = [], []
    for d in docs:
        x, m = candidates(d, M[d['doc_id']], refmap, priors, d['doc_id'] if excl else None, labels, TV)
        if len(x): X.append(x); meta += m
    return (np.vstack(X) if X else np.zeros((0, 1))), meta

## 5. Run: out-of-fold NER mentions (train+dev) and final NER for test

In [10]:
K = 5
folds = [allx[i::K] for i in range(K)]
MF = CACHE + 'mentions.pkl'
if os.path.exists(MF):
    M = pickle.load(open(MF, 'rb'))
else:
    M = {}; t0 = time.time()
    for fi in range(K):
        trn = [d for j, f in enumerate(folds) if j != fi for d in f]
        crf, lex = train_crf(trn)
        for d in folds[fi]: M[d['doc_id']] = predict_mentions(crf, lex, d)
        print('fold', fi, round(time.time() - t0), 's')
    crf, lex = train_crf(allx)               # final NER trained on everything
    for d in te: M[d['doc_id']] = predict_mentions(crf, lex, d)
    pickle.dump(M, open(MF, 'wb'))

fold 0 21 s
fold 1 42 s
fold 2 62 s
fold 3 84 s
fold 4 107 s


In [11]:
MUF = CACHE + 'mentions_u.pkl'
if os.path.exists(MUF):
    MU = pickle.load(open(MUF, 'rb'))
else:
    MU = {}
    for fi in range(K):
        trn = [d for j, f in enumerate(folds) if j != fi for d in f]
        st = form_stats(trn)
        for d in folds[fi]: MU[d['doc_id']] = union(M[d['doc_id']], lex_mentions(d, st))
    st = form_stats(allx)
    for d in te: MU[d['doc_id']] = union(M[d['doc_id']], lex_mentions(d, st))
    pickle.dump(MU, open(MUF, 'wb'))
for name, MM in [('CRF', M), ('CRF+lexicon', MU)]:      # mention-level (strict span) quality, OOF
    tp = fp = fn = 0
    for d in allx:
        G = {(e['offsets'][0][0], e['offsets'][-1][1], e['type']) for e in d['text_bound_annotations']['entities']}
        P = {(m['start'], m['end'], m['type']) for m in MM[d['doc_id']]}
        tp += len(G & P); fp += len(P - G); fn += len(G - P)
    p, r = tp / (tp + fp), tp / (tp + fn); print(name, 'P%.3f R%.3f F1 %.3f' % (p, r, 2 * p * r / (p + r)))

CRF P0.797 R0.648 F1 0.715
CRF+lexicon P0.780 R0.657 F1 0.713


## 6. Cross-validated relation scoring + evaluation with the official metric

In [12]:
oof = []
for fi in range(K):
    trn = [d for j, f in enumerate(folds) if j != fi for d in f]
    refmap = build_ref_map(trn); pri = Priors(trn)
    Xt, mt = build(trn, MU, pri, refmap); yt = np.array([m[4] for m in mt])
    Xv, mv = build(folds[fi], MU, pri, refmap, excl=False)
    oof += list(zip(mv, fit(Xt, yt).predict(Xv))); print('fold', fi, Xt.shape)
by = collections.defaultdict(list)
for m, p in oof: by[m[0]].append(((m[2], m[1], m[3]), p))
def thr_preds(t): return {k: [e for e, p in v if p >= t] for k, v in by.items()}
print('fixed threshold :', {t: round(score(thr_preds(t), allx), 4) for t in (0.25, 0.3, 0.35, 0.4)})
for rho in (1.0, 0.85, 0.7, 0.55):
    P = {k: select(v, rho, 0.3) for k, v in by.items()}
    print('expected-F1 rho=%.2f  all %.4f | dev %.4f' % (rho, score(P, allx), score(P, dv)))

fold 0 (7313, 154)
fold 1 (7782, 154)
fold 2 (7531, 154)
fold 3 (7539, 154)
fold 4 (8120, 154)
fixed threshold : {0.25: 0.5419, 0.3: 0.5319, 0.35: 0.5275, 0.4: 0.5142}
expected-F1 rho=1.00  all 0.5338 | dev 0.5535
expected-F1 rho=0.85  all 0.5385 | dev 0.5556
expected-F1 rho=0.70  all 0.5428 | dev 0.5597
expected-F1 rho=0.55  all 0.5429 | dev 0.5590


## 7. Final model (train+dev) → test predictions → `submission.csv`
Format: one row per test doc; `knowledge_graph` is a JSON list of `{predicate, subject, object}`.

In [13]:
import csv
refmap = build_ref_map(allx); pri = Priors(allx)
Xt, mt = build(allx, MU, pri, refmap); yt = np.array([m[4] for m in mt])
Xs, ms = build(te, MU, pri, refmap, labels=False, excl=False)
p = sum(fit(Xt, yt, seed=s).predict(Xs) for s in range(5)) / 5       # 5-seed average
byt = collections.defaultdict(list)
for m, pp in zip(ms, p): byt[m[0]].append(((m[2], m[1], m[3]), pp))
ne = nt = 0
with open(OUT + 'submission.csv', 'w', newline='', encoding='utf-8') as f:
    w = csv.writer(f); w.writerow(['doc_id', 'knowledge_graph'])
    for d in te:
        seen = set(); kg = []
        for s, pr, o in select(byt.get(d['doc_id'], []), rho=0.7, empty_t=0.3):
            k = (s.lower(), pr, o.lower())
            if k in seen: continue
            seen.add(k); kg.append({'predicate': pr, 'subject': s, 'object': o})
        ne += (not kg); nt += len(kg)
        w.writerow([d['doc_id'], json.dumps(kg, ensure_ascii=False)])
print('docs', len(te), '| empty graphs', ne, '| triples', nt, '| mean/doc %.1f' % (nt / len(te)))

docs 82 | empty graphs 5 | triples 1013 | mean/doc 12.4


In [14]:
# sanity-check the file
import pandas as pd
sub = pd.read_csv(OUT + 'submission.csv')
assert list(sub.doc_id.astype(str)) == [d['doc_id'] for d in te]
assert all(set(t) == {'predicate','subject','object'} for kg in sub.knowledge_graph for t in json.loads(kg))
print('OK', sub.shape); sub.head()

OK (82, 2)


,doc_id,knowledge_graph
0,104657,"[{""predicate"": ""Located_in"", ""subject"": ""Asian..."
1,107116,"[{""predicate"": ""Located_in"", ""subject"": ""banan..."
2,97985,"[{""predicate"": ""Located_in"", ""subject"": ""Xylel..."
3,103005,"[{""predicate"": ""Found_on"", ""subject"": ""Xylella..."
4,107265,"[{""predicate"": ""Located_in"", ""subject"": ""Fusar..."
